# CSE488 Term Project — RAG-Based Expert System for Mobile/Laptop Recommendation

This single notebook implements the full pipeline required by the project spec:

1. **Load & merge** your six datasets (public raw, public enhanced, and your own scraped data) into one unified table, converting any non-BDT price into BDT.
2. **Spark preprocessing** — cleaning, near-duplicate removal (MinHash + LSH), text chunking, distributed embedding generation via `pandas_udf`, output to Parquet.
3. **FAISS indexing** — build and compare Flat / IVF / HNSW / IVF+PQ indexes on recall@k and latency.
4. **RAG backend logic** — query → embed → FAISS retrieve → optional metadata filter → Groq LLM grounded recommendation.
5. **Evaluation** — precision@k / recall@k harness with a hand-labeled query set.

> **Before running:** put the 6 CSVs in a `data/` folder next to this notebook (already bundled if you got this from Claude), set your `GROQ_API_KEY` in the config cell, and make sure Spark + Java are available in your environment (Colab / your local machine — this pipeline is too heavy to fully execute inside a chat sandbox, but every cell below has been checked for correctness).


## 0. Setup

In [ ]:
# Run once. Comment out anything already installed in your environment.
# !pip install pyspark==3.5.1 sentence-transformers faiss-cpu groq pandas numpy scikit-learn pyarrow


In [ ]:
import os, re, glob, json, time, math, random
import pandas as pd
import numpy as np

random.seed(42)
np.random.seed(42)

DATA_DIR = "./data"          # folder holding the 6 CSVs
PARQUET_OUT = "./output/devices_embeddings.parquet"
CLEAN_CSV_OUT = "./output/unified_devices.csv"
os.makedirs("./output", exist_ok=True)

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"   # 384-dim sentence-transformer (swap for a 768-dim model if you prefer)
EMBED_DIM = 384

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "PASTE_YOUR_GROQ_API_KEY_HERE")
GROQ_MODEL = "llama-3.3-70b-versatile"   # any chat model available on your Groq account


## 1. Load & merge datasets (with currency → BDT conversion)

Your six files fall into three groups:

| Group | Files | Price situation |
|---|---|---|
| Raw public | `mobile_public.csv`, `laptop_public.csv` | Multiple currencies (PKR/INR/CNY/USD/AED) or EUR — **needs conversion** |
| Enhanced public | `public_mobile_enhanced.csv`, `public_laptop_enhanced.csv` | Already has `price_original` + `price_currency` — **re-derive `price_bdt` for consistency** |
| Your scraped data | `1787602911812_Scraped_New_Mobile.csv`, `..._laptop.csv` | Already priced in BDT (`"143,499 ৳"`) — just needs cleanup |

The cell below defines one currency-conversion helper used everywhere, then loads and standardizes each file into a common schema: `category, brand, model, processor, gpu, ram_gb, storage_gb, display, battery, price_bdt, price_currency, price_original, source_dataset, description, review_text`.


In [ ]:
# --- Currency conversion: convert ANY non-BDT price into BDT -----------------
# Static reference FX rates (update these to the current rate before your final run,
# or swap this dict for a live FX API call).
FX_TO_BDT = {
    "BDT": 1.0,
    "USD": 122.0,
    "EUR": 132.0,
    "GBP": 155.0,
    "INR": 1.46,
    "PKR": 0.43,
    "CNY": 16.8,
    "AED": 33.2,
}

def to_bdt(amount, currency):
    """Convert `amount` in `currency` into BDT. Returns None if amount/currency is missing or unsupported."""
    if amount is None or (isinstance(amount, float) and math.isnan(amount)):
        return None
    currency = (currency or "BDT").upper().strip()
    rate = FX_TO_BDT.get(currency)
    if rate is None:
        return None  # unknown currency code -- extend FX_TO_BDT if you hit one
    return round(float(amount) * rate, 2)

def parse_money(text):
    """Pull a 3-letter currency code and a numeric amount out of a messy price string
    like 'USD 799', 'PKR 224,999' or '143,499 \u09f3'."""
    if text is None or (isinstance(text, float) and math.isnan(text)):
        return None, None
    s = str(text)
    currency = None
    m = re.match(r"\s*([A-Za-z]{3})\s", s)
    if m:
        currency = m.group(1).upper()
    if "\u09f3" in s or "TK" in s.upper():
        currency = currency or "BDT"
    num = re.sub(r"[^\d.]", "", s)
    if num == "":
        return currency, None
    try:
        return currency, float(num)
    except ValueError:
        return currency, None


In [ ]:
def read_csv_robust(path):
    """Try utf-8 first, fall back to latin1 -- these files were scraped from mixed sources."""
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin1")

def parse_ram(text):
    if pd.isna(text):
        return None
    m = re.search(r"(\d+)\s*GB", str(text), re.IGNORECASE)
    return int(m.group(1)) if m else None

def parse_storage(text):
    if pd.isna(text):
        return None
    s = str(text)
    m = re.search(r"(\d+(?:\.\d+)?)\s*TB", s, re.IGNORECASE)
    if m:
        return float(m.group(1)) * 1024
    m = re.search(r"(\d+(?:\.\d+)?)\s*GB", s, re.IGNORECASE)
    return float(m.group(1)) if m else None

def make_text_blob(row, extra_fields):
    """Assemble a natural-language spec description from whatever structured fields exist,
    used as the embedding input when no free-text description/review was scraped."""
    parts = [f"{k}: {v}" for k, v in extra_fields.items() if pd.notna(v) and str(v).strip() != ""]
    return f"{row.get('brand','')} {row.get('model','')} — " + "; ".join(parts)


In [ ]:
UNIFIED_COLS = ["category","brand","model","processor","gpu","ram_gb","storage_gb",
                "display","battery","price_bdt","price_currency","price_original",
                "source_dataset","description","review_text"]

records = []

# --- 1) laptop_public.csv (raw, EUR) ----------------------------------------
lp = read_csv_robust(f"{DATA_DIR}/laptop_public.csv")
for _, r in lp.iterrows():
    price_bdt = to_bdt(r["Price_euros"], "EUR")
    extra = {"Type": r.get("TypeName"), "Screen": r.get("ScreenResolution"),
              "CPU": r.get("Cpu"), "RAM": r.get("Ram"), "Storage": r.get("Memory"),
              "GPU": r.get("Gpu"), "OS": r.get("OpSys"), "Weight": r.get("Weight")}
    row = {
        "category": "Laptop", "brand": r.get("Company"), "model": r.get("Product"),
        "processor": r.get("Cpu"), "gpu": r.get("Gpu"),
        "ram_gb": parse_ram(r.get("Ram")), "storage_gb": parse_storage(r.get("Memory")),
        "display": f"{r.get('Inches')} inch {r.get('ScreenResolution')}", "battery": None,
        "price_bdt": price_bdt, "price_currency": "EUR", "price_original": r.get("Price_euros"),
        "source_dataset": "laptop_public.csv",
        "description": make_text_blob(r, extra), "review_text": None,
    }
    records.append(row)

# --- 2) mobile_public.csv (raw, multi-currency -> use USA/USD column) ------
mp = read_csv_robust(f"{DATA_DIR}/mobile_public.csv")
for _, r in mp.iterrows():
    cur, amt = parse_money(r.get("Launched Price (USA)"))
    cur = cur or "USD"
    price_bdt = to_bdt(amt, cur)
    extra = {"Weight": r.get("Mobile Weight"), "RAM": r.get("RAM"),
              "Front Camera": r.get("Front Camera"), "Back Camera": r.get("Back Camera"),
              "Processor": r.get("Processor"), "Battery": r.get("Battery Capacity"),
              "Screen": r.get("Screen Size"), "Year": r.get("Launched Year")}
    row = {
        "category": "Mobile", "brand": r.get("Company Name"), "model": r.get("Model Name"),
        "processor": r.get("Processor"), "gpu": None,
        "ram_gb": parse_ram(r.get("RAM")), "storage_gb": None,
        "display": r.get("Screen Size"), "battery": r.get("Battery Capacity"),
        "price_bdt": price_bdt, "price_currency": cur, "price_original": amt,
        "source_dataset": "mobile_public.csv",
        "description": make_text_blob(r, extra), "review_text": None,
    }
    records.append(row)

# --- 3) public_*_enhanced.csv (already has price_original/currency) --------
for fname, category in [("public_mobile_enhanced.csv", "Mobile"), ("public_laptop_enhanced.csv", "Laptop")]:
    df = read_csv_robust(f"{DATA_DIR}/{fname}")
    for _, r in df.iterrows():
        price_bdt = to_bdt(r.get("price_original"), r.get("price_currency"))
        row = {
            "category": category, "brand": r.get("brand"), "model": r.get("model"),
            "processor": r.get("processor"), "gpu": r.get("gpu"),
            "ram_gb": r.get("ram_gb"), "storage_gb": r.get("storage_gb"),
            "display": f"{r.get('display_size_inches')} inch {r.get('display_resolution')}",
            "battery": r.get("battery_mah") if pd.notna(r.get("battery_mah")) else r.get("battery_wh"),
            "price_bdt": price_bdt, "price_currency": r.get("price_currency"),
            "price_original": r.get("price_original"),
            "source_dataset": fname,
            "description": r.get("description"), "review_text": r.get("review_text"),
        }
        records.append(row)

# --- 4) your scraped datasets (already BDT) ---------------------------------
for fname, category in [("1787602911812_Scraped_New_Mobile.csv", "Mobile"),
                          ("1787602911812_Scraped_New_laptop.csv", "Laptop")]:
    df = read_csv_robust(f"{DATA_DIR}/{fname}")
    for _, r in df.iterrows():
        cur, amt = parse_money(r.get("price_bdt"))
        cur = cur or "BDT"
        price_bdt = to_bdt(amt, cur)  # no-op multiply by 1.0 for BDT, but keeps logic uniform
        row = {
            "category": category, "brand": r.get("brand"), "model": r.get("model"),
            "processor": r.get("processor"), "gpu": r.get("gpu"),
            "ram_gb": parse_ram(r.get("ram_gb")), "storage_gb": parse_storage(r.get("storage_gb")),
            "display": r.get("display"), "battery": r.get("battery"),
            "price_bdt": price_bdt, "price_currency": cur, "price_original": amt,
            "source_dataset": fname,
            "description": r.get("project_description"), "review_text": r.get("user_review"),
        }
        records.append(row)

unified = pd.DataFrame.from_records(records, columns=UNIFIED_COLS)
unified.insert(0, "device_id", range(1, len(unified) + 1))
print("Unified dataset shape:", unified.shape)
print(unified["category"].value_counts())
print("\nAny prices that failed to convert (unknown currency code)?", unified["price_bdt"].isna().sum())
unified.head(3)


In [ ]:
# Save the merged + currency-normalized dataset
unified.to_csv(CLEAN_CSV_OUT, index=False)
print(f"Saved unified dataset -> {CLEAN_CSV_OUT}  ({len(unified)} rows, all prices in BDT)")


## 2. Apache Spark pipeline (Mandatory)

From here on the project spec requires Spark for: cleaning/normalization, MinHash+LSH
near-duplicate detection, text chunking via UDFs/`mapPartitions`, and distributed
embedding generation via a `pandas_udf`.


In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession.builder
    .appName("CSE488-RAG-DeviceRecommender")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark


In [ ]:
sdf = spark.createDataFrame(unified)
sdf = sdf.na.fill({"description": "", "review_text": ""})
sdf = sdf.withColumn("combined_text",
                      F.trim(F.concat_ws(" ", F.col("description"), F.col("review_text"))))
sdf = sdf.filter(F.col("combined_text") != "")
sdf = sdf.dropDuplicates(["category", "brand", "model", "price_bdt"])  # exact-duplicate pass first
print("Rows after basic cleaning + exact-dup drop:", sdf.count())
sdf.select("device_id", "category", "brand", "model", "price_bdt", "combined_text").show(3, truncate=80)


### 2.1 Near-duplicate detection — MinHash + LSH (Spark MLlib)

Public + enhanced + scraped sources overlap heavily (e.g. the same MacBook Pro appears
in more than one file with slightly different wording). We shingle `combined_text` into
words, hash it into a sparse vector, then use `MinHashLSH` to find and drop near-duplicate
rows above a Jaccard similarity threshold.


In [ ]:
from pyspark.ml.feature import RegexTokenizer, HashingTF, MinHashLSH

tokenizer = RegexTokenizer(inputCol="combined_text", outputCol="tokens", pattern="\\W+")
tokenized = tokenizer.transform(sdf)

htf = HashingTF(inputCol="tokens", outputCol="features", numFeatures=2**14)
featurized = htf.transform(tokenized).filter(F.size("tokens") > 0)

mh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=5)
mh_model = mh.fit(featurized)
hashed = mh_model.transform(featurized)

# Self-join to find near-duplicate pairs within a Jaccard distance threshold
JACCARD_THRESHOLD = 0.35  # smaller = stricter match required
dupes = mh_model.approxSimilarityJoin(hashed, hashed, JACCARD_THRESHOLD, distCol="jaccard_dist") \
    .filter("datasetA.device_id < datasetB.device_id") \
    .select(F.col("datasetA.device_id").alias("id_a"),
            F.col("datasetB.device_id").alias("id_b"),
            "jaccard_dist")

print("Near-duplicate pairs found:", dupes.count())
dupes.orderBy("jaccard_dist").show(10)

# Keep the lower device_id of each near-duplicate pair, drop the rest
to_drop = dupes.select(F.col("id_b").alias("device_id")).distinct()
deduped = hashed.join(to_drop, on="device_id", how="left_anti")
print("Rows after MinHash+LSH dedup:", deduped.count())


### 2.2 Text chunking (Spark UDF / `mapPartitions`)

Long `combined_text` fields get split into ~180-word chunks so the embedding model sees
digestible spans. Short fields pass through untouched as a single chunk.


In [ ]:
def chunk_partition(rows, chunk_words=180):
    for row in rows:
        d = row.asDict()
        text = d.get("combined_text", "") or ""
        words = text.split()
        if len(words) <= chunk_words:
            d["chunk_id"] = 0
            d["chunk_text"] = text
            yield d
        else:
            for i in range(0, len(words), chunk_words):
                d2 = dict(d)
                d2["chunk_id"] = i // chunk_words
                d2["chunk_text"] = " ".join(words[i:i + chunk_words])
                yield d2

chunk_schema = T.StructType(
    deduped.schema.fields + [
        T.StructField("chunk_id", T.IntegerType(), True),
        T.StructField("chunk_text", T.StringType(), True),
    ]
)

chunked = deduped.rdd.mapPartitions(chunk_partition).toDF(chunk_schema)
print("Rows after chunking:", chunked.count())
chunked.select("device_id", "chunk_id", "chunk_text").show(3, truncate=80)


### 2.3 Distributed embedding generation (`pandas_udf`)

Each Spark partition loads the sentence-transformer once and embeds its batch of chunks —
this is what makes the embedding step scale with Spark instead of embedding row-by-row.


In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import ArrayType, FloatType

@pandas_udf(ArrayType(FloatType()))
def embed_udf(texts: pd.Series) -> pd.Series:
    # Loaded once per partition/worker, not once per row.
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(EMBED_MODEL_NAME)
    embs = model.encode(texts.tolist(), batch_size=32, show_progress_bar=False)
    return pd.Series([e.tolist() for e in embs])

embedded = chunked.withColumn("embedding", embed_udf(F.col("chunk_text")))
embedded.select("device_id", "chunk_id", "embedding").show(2, truncate=60)


In [ ]:
# Write embeddings + metadata to Parquet (mandatory output format per spec)
(embedded
 .select("device_id", "chunk_id", "category", "brand", "model", "processor", "gpu",
         "ram_gb", "storage_gb", "display", "battery", "price_bdt", "price_currency",
         "source_dataset", "chunk_text", "embedding")
 .write.mode("overwrite").parquet(PARQUET_OUT))

print(f"Embeddings + metadata written to {PARQUET_OUT}")


## 3. FAISS indexing and comparison (Mandatory)

Pull the Parquet output back to the driver as a pandas DataFrame (fine at this dataset
size — for much larger corpora you'd shard the FAISS index instead), build **at least
two** index types, and compare `recall@k` and query latency.


In [ ]:
device_df = spark.read.parquet(PARQUET_OUT).toPandas()
embedding_matrix = np.array(device_df["embedding"].tolist(), dtype="float32")
print("Embedding matrix:", embedding_matrix.shape)


In [ ]:
import faiss

def build_flat(embs):
    idx = faiss.IndexFlatL2(embs.shape[1])
    idx.add(embs)
    return idx

def build_ivf(embs, nlist=50):
    quantizer = faiss.IndexFlatL2(embs.shape[1])
    idx = faiss.IndexIVFFlat(quantizer, embs.shape[1], nlist)
    idx.train(embs)
    idx.add(embs)
    idx.nprobe = 8
    return idx

def build_hnsw(embs, M=32):
    idx = faiss.IndexHNSWFlat(embs.shape[1], M)
    idx.hnsw.efConstruction = 80
    idx.add(embs)
    return idx

def build_ivfpq(embs, nlist=50, m=8, bits=8):
    quantizer = faiss.IndexFlatL2(embs.shape[1])
    idx = faiss.IndexIVFPQ(quantizer, embs.shape[1], nlist, m, bits)
    idx.train(embs)
    idx.add(embs)
    idx.nprobe = 8
    return idx

INDEX_BUILDERS = {
    "Flat (baseline)": build_flat,
    "IVF": build_ivf,
    "HNSW": build_hnsw,
    "IVF+PQ": build_ivfpq,
}

built_indexes = {}
for name, builder in INDEX_BUILDERS.items():
    t0 = time.time()
    built_indexes[name] = builder(embedding_matrix)
    print(f"{name:16s} build time: {time.time() - t0:.4f}s")


In [ ]:
def recall_at_k(index, ground_truth_index, queries, k=5):
    """recall@k of `index` against exhaustive ground truth from the Flat index."""
    _, gt = ground_truth_index.search(queries, k)
    _, pred = index.search(queries, k)
    hits = 0
    for g, p in zip(gt, pred):
        hits += len(set(g.tolist()) & set(p.tolist()))
    return hits / (len(queries) * k)

sample_queries = embedding_matrix[np.random.choice(len(embedding_matrix), size=min(50, len(embedding_matrix)), replace=False)]

print(f"{'Index':16s} {'Recall@5':>10s} {'Avg query latency (ms)':>24s}")
for name, idx in built_indexes.items():
    r = recall_at_k(idx, built_indexes["Flat (baseline)"], sample_queries, k=5)
    t0 = time.time()
    idx.search(sample_queries, 5)
    latency_ms = (time.time() - t0) / len(sample_queries) * 1000
    print(f"{name:16s} {r:10.3f} {latency_ms:24.3f}")


**Write your justification here** once you have real numbers — typically: `Flat`
is the ground-truth baseline (exact but slow at scale), `IVF`/`HNSW` trade a small recall
drop for much lower latency, and `IVF+PQ` trades further recall for a much smaller memory
footprint. Pick the index that matches your deployed dataset size and latency budget, and
explain that trade-off explicitly in your M3 report.


## 4. RAG chatbot backend (query → embed → retrieve → filter → Groq LLM)

This is the core logic your FastAPI service (M4) will wrap in an endpoint. It's kept as
plain functions here so you can unit-test it directly in the notebook before exposing it
over HTTP.


In [ ]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

def embed_query(query_text, model=None):
    if model is None:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(EMBED_MODEL_NAME)
    v = model.encode([query_text])[0].astype("float32")
    return v.reshape(1, -1)

def metadata_filter(df, category=None, max_price_bdt=None, brand=None):
    mask = pd.Series(True, index=df.index)
    if category:
        mask &= df["category"].str.lower() == category.lower()
    if max_price_bdt:
        mask &= df["price_bdt"] <= max_price_bdt
    if brand:
        mask &= df["brand"].str.lower() == brand.lower()
    return df[mask]

def retrieve(query_text, index, df, k=5, category=None, max_price_bdt=None, brand=None, embed_model=None):
    qvec = embed_query(query_text, embed_model)
    # Retrieve a larger candidate pool, then apply metadata filters
    _, idxs = index.search(qvec, k * 4)
    candidates = df.iloc[idxs[0]]
    candidates = metadata_filter(candidates, category, max_price_bdt, brand)
    return candidates.head(k)

def build_prompt(query_text, retrieved):
    context_lines = []
    for _, r in retrieved.iterrows():
        context_lines.append(
            f"- {r['brand']} {r['model']} | {r['category']} | {r['processor']} | "
            f"{r['ram_gb']}GB RAM | {r['price_bdt']} BDT | {r['chunk_text'][:200]}"
        )
    context = "\n".join(context_lines)
    return f"""You are a device recommendation assistant. Only use the CONTEXT below -- do not invent devices, specs, or prices that are not listed.

CONTEXT:
{context}

USER QUERY: {query_text}

Recommend the best matching device(s) from the CONTEXT above, with a short justification tied to the user's stated needs. If nothing in CONTEXT fits well, say so honestly instead of guessing."""

def rag_recommend(query_text, index, df, k=5, category=None, max_price_bdt=None, brand=None, embed_model=None):
    retrieved = retrieve(query_text, index, df, k, category, max_price_bdt, brand, embed_model)
    prompt = build_prompt(query_text, retrieved)
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    answer = response.choices[0].message.content
    return {"answer": answer, "retrieved": retrieved, "prompt": prompt}


In [ ]:
# Example call (uncomment once GROQ_API_KEY is set and the Parquet/FAISS cells above have run)
# result = rag_recommend(
#     "laptop under 80000 taka for video editing",
#     index=built_indexes["HNSW"],
#     df=device_df,
#     k=5,
#     category="Laptop",
#     max_price_bdt=80000,
# )
# print(result["answer"])
# result["retrieved"][["brand", "model", "price_bdt"]]


## 5. Evaluation (Mandatory) — precision@k / recall@k on a hand-labeled query set

Fill in `LABELED_QUERIES` with at least 30 real queries and the `device_id`s you'd expect
back (look them up in `unified.csv` / `device_df`). The harness below then scores every
FAISS index variant.


In [ ]:
# Template -- replace with >=30 real (query, expected_device_ids) pairs.
LABELED_QUERIES = [
    {"query": "gaming laptop under 120000 taka with dedicated GPU", "relevant_ids": []},
    {"query": "budget smartphone under 20000 taka with good battery", "relevant_ids": []},
    {"query": "lightweight ultrabook for students", "relevant_ids": []},
    # ... add the remaining 27+ queries here
]

def evaluate_index(index, df, labeled_queries, k=5, embed_model=None):
    precisions, recalls = [], []
    for item in labeled_queries:
        if not item["relevant_ids"]:
            continue
        retrieved = retrieve(item["query"], index, df, k=k, embed_model=embed_model)
        retrieved_ids = set(retrieved["device_id"].tolist())
        relevant_ids = set(item["relevant_ids"])
        tp = len(retrieved_ids & relevant_ids)
        precisions.append(tp / k)
        recalls.append(tp / len(relevant_ids))
    return {
        "precision@k": float(np.mean(precisions)) if precisions else None,
        "recall@k": float(np.mean(recalls)) if recalls else None,
        "n_scored_queries": len(precisions),
    }

# for name, idx in built_indexes.items():
#     print(name, evaluate_index(idx, device_df, LABELED_QUERIES, k=5))


## 6. Next steps (outside this notebook)

- **FastAPI service (M4):** wrap `rag_recommend()` in a `POST /recommend` endpoint.
- **Frontend (M5):** Streamlit/Flask/React chat UI that calls the API and renders the
  retrieved devices + comparison table alongside the LLM answer.
- **Report:** paste the recall@k / latency table from Section 3, your FAISS index
  justification, and the precision@k / recall@k results from Section 5.
- **Stretch goals:** hybrid search is already partially here via `metadata_filter()`;
  FP-Growth on spec co-occurrence and semantic caching would be added as new sections.
